# 01 · Explore source data
**Phase 2 – Data profiling.** Understand the 10 raw Parquet tables in the `source` container before building Bronze.
This notebook only **reads** data; it creates no tables.


## 1. Check access to the source container
Lists the table folders in the data lake through the Unity Catalog external location (no keys).

In [0]:
SOURCE = "abfss://source@stglovolake122.dfs.core.windows.net"
display(dbutils.fs.ls(SOURCE))

## 2. Load all tables and check size
Reads each folder into a DataFrame and compares row and column counts with the data dictionary.

In [0]:
tables = [
    "cities", "couriers", "cuisine_category_translation", "customers", "menu_items",
    "order_items", "orders", "payments", "restaurants", "reviews",
]

dfs = {t: spark.read.parquet(f"{SOURCE}/{t}/") for t in tables}

for t, df in dfs.items():
    print(f"{t:30} {df.count():>9,} rows   {len(df.columns)} columns")

In [0]:
dfs.keys()

## 3. Columns and data types
Shows each table's columns and their types, to spot values stored as the wrong type (e.g. prices stored as text).

In [0]:
for t in tables:
    print(f"===== {t} =====")
    dfs[t].printSchema()

## 4. Missing values
Counts empty (null) values per column, so we know which columns need rules for missing data.

In [0]:
from pyspark.sql import functions as F

for t, df in dfs.items():
    print(f"===== {t} =====")
    null_counts = df.select(
        [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]
    )
    display(null_counts)

## 5. Are the nulls expected?
Checks missing courier and timestamps per order status: empty values are normal for cancelled/pending orders, but not for delivered ones.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW orders_parquet
USING parquet
OPTIONS (path 'abfss://source@stglovolake122.dfs.core.windows.net/orders/');

SELECT
  LOWER(TRIM(order_status)) AS status,
  COUNT(*) AS orders,
  SUM(CASE WHEN courier_id IS NULL THEN 1 ELSE 0 END)          AS no_courier,
  SUM(CASE WHEN order_picked_up_at IS NULL THEN 1 ELSE 0 END)  AS no_pickup_time,
  SUM(CASE WHEN order_delivered_at IS NULL THEN 1 ELSE 0 END)   AS no_delivery_time
FROM orders_parquet
GROUP BY LOWER(TRIM(order_status))
ORDER BY status

## 6. Duplicates
Compares total rows with distinct rows (exact copies) and distinct IDs (repeated keys).

In [0]:
keys = {
    "cities": "city_id",
    "couriers": "courier_id",
    "cuisine_category_translation": "cuisine_category_pl",
    "customers": "customer_id",
    "menu_items": "item_id",
    "order_items": "order_item_id",
    "orders": "order_id",
    "payments": None,          # no ID column in payments
    "restaurants": "restaurant_id",
    "reviews": "review_id",
}

for t in tables:
    df = dfs[t]
    total = df.count()
    exact_dupes = total - df.distinct().count()
    key = keys[t]
    key_dupes = total - df.select(key).distinct().count() if key else "-"
    print(f"{t:30} rows {total:>8,}   exact duplicates {exact_dupes:>6,}   repeated IDs {key_dupes}")

## 7. Inconsistent categories
Lists every distinct value and how often it appears, to find spelling variants that must be mapped to one standard value.

In [0]:
def show_values(table, column):
    print(f"===== {table}.{column} =====")
    display(
        dfs[table]
        .groupBy(column)
        .count()
        .orderBy(F.desc("count"))
    )

show_values("orders", "order_status")
show_values("payments", "payment_type")

### 7b. City names that don't match the official list
Compares the city written in restaurants, couriers and customers with the 36 official names in `cities`.

In [0]:
official = dfs["cities"].select(F.col("city_name").alias("city"))

def unmatched_cities(table, column):
    df = dfs[table].select(F.col(column).alias("city"))
    bad = df.join(official, on="city", how="left_anti")
    print(f"===== {table}.{column}: {bad.count():,} rows don't match exactly =====")
    display(bad.groupBy("city").count().orderBy(F.desc("count")))

unmatched_cities("restaurants", "city")
unmatched_cities("couriers", "city")
unmatched_cities("customers", "customer_city")

## 8. Links between tables (orphans)
Checks that every ID used in one table exists in the table it points to. Rows that point to a missing ID are orphans.

In [0]:
links = [
    # (child table,  child column,          parent table,                   parent column)
    ("orders",       "customer_id",         "customers",                    "customer_id"),
    ("orders",       "restaurant_id",       "restaurants",                  "restaurant_id"),
    ("orders",       "courier_id",          "couriers",                     "courier_id"),
    ("order_items",  "order_id",            "orders",                       "order_id"),
    ("order_items",  "item_id",             "menu_items",                   "item_id"),
    ("payments",     "order_id",            "orders",                       "order_id"),
    ("reviews",      "order_id",            "orders",                       "order_id"),
    ("menu_items",   "restaurant_id",       "restaurants",                  "restaurant_id"),
    ("restaurants",  "cuisine_category_pl", "cuisine_category_translation", "cuisine_category_pl"),
]

for child, child_col, parent, parent_col in links:
    ids_in_child = dfs[child].select(F.col(child_col).alias("id")).where(F.col("id").isNotNull())
    ids_in_parent = dfs[parent].select(F.col(parent_col).alias("id"))
    orphans = ids_in_child.join(ids_in_parent, on="id", how="left_anti").count()
    print(f"{child}.{child_col:20} → {parent:30} orphans: {orphans:,}")

## 9. Summary of findings → rules for Silver

| # | Problem | Where | Size | Silver rule |
|---|---|---|---|---|
| 1 | Values stored as text | `price_pln`, `payment_value_pln`, order timestamps | 3 columns | Cast prices to decimal; parse 4 timestamp formats |
| 2 | Inconsistent categories | `order_status` (16 variants), `payment_type` (18 variants) | ~16% of rows | Clean format, then map with a mapping table; unknown values → quarantine |
| 3 | City names don't match | restaurants, couriers, customers | ~2,170 rows | Match on a normalised key (trim, lowercase, remove Polish accents) → official name + `city_id` |
| 4 | Duplicates | order_items, payments, reviews, customers, restaurants | ~9,200 exact copies | Remove exact copies |
| 5 | Repeated order IDs | orders | 1,500 | Keep one row per `order_id` (earliest) |
| 6 | Unexpected nulls | orders delivered with no courier; missing dish category; courier city | ~1,900 rows | Quarantine / fill dish category from restaurant / `"Unknown"` city |
| 7 | Expected nulls | pickup/delivery times, review comments, ratings | – | Keep empty (never fill ratings with 0) |
| 8 | Orphans | orders, order_items, payments, reviews | ~11,300 rows | Quarantine with reason |